# Lab 2: Model Optimization Techniques - Part C: Knowledge Distillation

Welcome to lab 2 of the TinyML course! In this lab, we are going to learn a few approaches for deep learning model optimization. These techniques, when combined, can significantly shrink model size and make them suitable for model deployment. The techniques we are going to cover today are:

* Knowledge Distillation
* Model Pruning
* Quantization

Please follow Part A (Quantization) to set up the environment and install dependencies. At the beginning, we again apply the workaround from part A to resolve compatibility issues between `tensorflow`, `keras` and `tensorflow-model-optimization`:

In [1]:
%pip install numpy tensorflow-model-optimization tf_keras tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 whi

In [1]:
import os

# Apply workaround before imports
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

## Knowledge Distillation / Pruning / Quantization

This notebook demonstrates how to combine **model compression techniques**—Knowledge Distillation (KD), Pruning, and Quantization—to create compact and efficient deep learning models suitable for deployment on resource-constrained devices.

### 🧠 Knowledge Distillation (KD)
Knowledge Distillation is a technique where a smaller model (student) is trained to mimic a larger, pre-trained model (teacher). The student learns from the teacher's output probabilities (soft labels), providing richer supervision than traditional hard labels.

### ✂️ Pruning
Pruning removes less important connections (weights) from the student model, introducing sparsity to reduce model size and inference cost.

### 🧮 Quantization
Quantization reduces the numerical precision of weights and activations. In this notebook, we apply **Float16** or **Integer Quantization** to the pruned student model to further compress it.

### What This Notebook Demonstrates:
1. Training a **teacher model** on the MNIST dataset.
2. Training a smaller **student model** using **knowledge distillation**.
3. Applying **pruning** to the student model to introduce sparsity.
4. Applying **quantization** (e.g., Float16 or INT8) to the pruned student model.
5. Comparing model sizes and accuracies of:
   - Teacher Model  
   - Student Model    
   - Quantized + Pruned Student Model


In [2]:
# Import required libraries
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize the data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape the data for CNN input
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"Training data shape: {x_train.shape}, Labels shape: {y_train.shape}")
print(f"Test data shape: {x_test.shape}, Labels shape: {y_test.shape}")


11490434/11490434 [==============================] - 0s 0us/step
Training data shape: (60000, 28, 28, 1), Labels shape: (60000, 10)
Test data shape: (10000, 28, 28, 1), Labels shape: (10000, 10)


In [3]:
from tensorflow.data import Dataset
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Build a simple CNN model as the teacher
def create_teacher_model():
    return Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

# Create training and validation datasets
train_dataset = Dataset.from_tensor_slices((x_train, y_train))
val_dataset = Dataset.from_tensor_slices((x_test, y_test))
batch_size = 32

# Compile and train the model
teacher_model = create_teacher_model()
teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
teacher_history = teacher_model.fit(train_dataset.batch(batch_size), epochs=2, validation_data=val_dataset.batch(batch_size))

# Get validation accuracy
teacher_accuracy = teacher_history.history["val_accuracy"][-1]


Epoch 1/2
1875/1875 [==============================] - 11s 4ms/step - loss: 0.1578 - accuracy: 0.9538 - val_loss: 0.0812 - val_accuracy: 0.9717
Epoch 2/2
1875/1875 [==============================] - 7s 4ms/step - loss: 0.0533 - accuracy: 0.9836 - val_loss: 0.0471 - val_accuracy: 0.9836


## Knowledge Distillation

The student model is trained using Knowledge Distillation. Instead of directly using the ground-truth labels, the student learns from the teacher's output probabilities (soft labels) using the Kullback-Leibler (KL) divergence.

In [4]:
# Define the student model
def create_student_model():
    return Sequential([
        # Note we are using fewer channels here
        Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        MaxPooling2D((2, 2)),
        Flatten(),
        # Note we are using a smaller linear layer
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])

student_model = create_student_model()

# Precompute teacher predictions (soft labels)
temperature = 5
teacher_soft_labels = teacher_model.predict(x_train, batch_size=32) / temperature

# Combine ground-truth labels and teacher soft labels
teacher_labels_dataset = Dataset.from_tensor_slices(teacher_soft_labels)
kd_dataset = Dataset.zip((train_dataset, teacher_labels_dataset))

1875/1875 [==============================] - 3s 2ms/step


In [5]:
from tensorflow.keras import Model
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence
from tensorflow.keras.metrics import Mean
from tensorflow import GradientTape, nn

# Custom Knowledge Distillation training loss (used only during train_step)
def kd_loss(y_true, y_pred, soft_labels, temperature):
    # Hard label loss (ground truth)
    hard_loss = CategoricalCrossentropy()(y_true, y_pred)

    # Soft label loss (teacher predictions for the batch)
    soft_loss = KLDivergence()(
        nn.softmax(soft_labels / temperature),
        nn.softmax(y_pred / temperature)
    )

    # Standard KD formulation scales soft loss by temperature^2
    combined_loss = 0.5 * hard_loss + 0.5 * (temperature ** 2) * soft_loss

    return hard_loss, soft_loss, combined_loss

class KnowledgeDistiller(Model):
    def __init__(self, student, temperature):
        super().__init__()

        self.student = student
        self.temperature = temperature

        # Track each KD loss component per epoch
        self.hard_loss_tracker = Mean(name="hard_loss")
        self.soft_loss_tracker = Mean(name="soft_loss")
        self.kd_loss_tracker = Mean(name="kd_loss")

    @property
    def metrics(self):
        # Include compiled metrics + custom KD loss trackers
        return super().metrics + [
            self.hard_loss_tracker,
            self.soft_loss_tracker,
            self.kd_loss_tracker,
        ]

    def call(self, inputs, training=False):
        return self.student(inputs, training=training)

    def train_step(self, data):
        (x_batch, y_batch), soft_labels = data

        # Forward pass: compute KD losses
        with GradientTape() as tape:
            y_pred = self(x_batch, training=True)
            hard_loss, soft_loss, combined_loss = kd_loss(
                y_batch, y_pred, soft_labels, temperature=self.temperature
            )

        # Backward pass: compute gradients and weight updates
        grads = tape.gradient(combined_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        # Update custom loss trackers
        self.hard_loss_tracker.update_state(hard_loss)
        self.soft_loss_tracker.update_state(soft_loss)
        self.kd_loss_tracker.update_state(combined_loss)

        # Update compiled metrics (e.g., accuracy) using non-deprecated path
        metric_results = self.compute_metrics(
            x_batch, y_batch, y_pred, sample_weight=None
        )

        return {
            **metric_results,
            "loss": self.kd_loss_tracker.result(),
            "hard_loss": self.hard_loss_tracker.result(),
            "soft_loss": self.soft_loss_tracker.result()
        }

# Compile and train the student model in the wrapper
# (Note: loss only applies for evaluation because we overrode it in `train_step`)
distiller = KnowledgeDistiller(student_model, temperature=temperature)
distiller.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
distiller.fit(kd_dataset.batch(batch_size), epochs=2, verbose=1)

Epoch 1/2
1875/1875 [==============================] - 14s 5ms/step - accuracy: 0.9396 - hard_loss: 0.2063 - soft_loss: 0.0011 - kd_loss: 0.1165 - loss: 0.2024
Epoch 2/2
1875/1875 [==============================] - 9s 5ms/step - accuracy: 0.9782 - hard_loss: 0.0741 - soft_loss: 0.0012 - kd_loss: 0.0518 - loss: 0.0566


In [19]:
# Evaluate the student model
student_accuracy = distiller.evaluate(val_dataset.batch(batch_size), verbose=0)[1]
print(f"Student Model Accuracy: {student_accuracy:.4f}")

Student Model Accuracy: 0.9762


---

### ✂️➕🔢 Pruning + Quantization Task

Now that you have successfully trained the **student model using Knowledge Distillation**, your next task is to **compress** it further for deployment.

Using your knowledge from the previous tasks related to quantization and pruning:

1. **Apply pruning** to the student model to introduce sparsity.
2. **Convert** the pruned model to TensorFlow Lite format using **full integer (INT8) quantization**.
   - Be sure to include a **representative dataset** for calibration.
   - Use appropriate settings to ensure the model is both **sparse** and **fully quantized**.

Finally:

- **Save your model** as `pruned_quantized_student_model.tflite`.
- This version should be **smaller** and suitable for deployment on **resource-constrained hardware**.

---


---

### 🧪 Lab 2 Submission Reminder

**Please complete the code below and take a screenshot of it as part of your Lab 2 submission.**

⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️

---

In [8]:
# [ TODO ]
# 1. Apply pruning to the trained student model
# <--- Your code here to prune the student model --->
from tensorflow_model_optimization.sparsity.keras import (
    prune_low_magnitude,
    PolynomialDecay,
    UpdatePruningStep
)

# Apply pruning to the model using a polynomial decay schedule
def apply_pruning(model):
    pruning_params = {
        'pruning_schedule': PolynomialDecay(
            initial_sparsity=0.5,   # Start pruning with 50% of weights set to zero
            final_sparsity=0.9,     # Gradually increase sparsity to 90% by end_step
            begin_step=0,           # Start pruning from the first training step
            end_step=2000           # End pruning at step 2000
        )
    }
    pruned_model = prune_low_magnitude(model, **pruning_params)
    return pruned_model

# Compile and train the pruned model
pruned_model = apply_pruning(student_model)
pruned_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Add the pruning update callback (required)
callbacks = [UpdatePruningStep()]

# Train the pruned model
history = pruned_model.fit(
    train_dataset.batch(batch_size),
    epochs=2,
    batch_size=32,
    validation_data=val_dataset.batch(batch_size),
    callbacks=callbacks  # Important: ensures pruning step is updated during training
)

# Get accuracy of the pruned model
pruned_accuracy = history.history["val_accuracy"][-1]
print(f"Pruned Model Accuracy: {pruned_accuracy:.4f}")


# raise NotImplementedError


Epoch 1/2
1875/1875 [==============================] - 15s 6ms/step - loss: 0.0789 - accuracy: 0.9763 - val_loss: 0.0839 - val_accuracy: 0.9742
Epoch 2/2
1875/1875 [==============================] - 14s 7ms/step - loss: 0.0687 - accuracy: 0.9791 - val_loss: 0.0786 - val_accuracy: 0.9762
Pruned Model Accuracy: 0.9762


In [16]:
from tensorflow import lite as tf_lite
from tensorflow_model_optimization.sparsity.keras import strip_pruning

# Strip the pruning mask
stripped_model = strip_pruning(pruned_model)

# # Convert the stripped model with experimental sparsity optimization
# converter = tf_lite.TFLiteConverter.from_keras_model(stripped_model)
# converter.optimizations = [tf_lite.Optimize.EXPERIMENTAL_SPARSITY]
# tflite_model_sparse = converter.convert()

# Save the optimized sparse TFLite model
with open("pruned_model_sparse.tflite", "wb") as f:
    f.write(tflite_model_sparse)

print("Saved pruned model with experimental sparsity optimization.")

Saved pruned model with experimental sparsity optimization.


In [17]:

# 2. Convert the pruned model to TFLite using full integer quantization
# (Don't forget to provide a representative dataset and use appropriate settings for optimizations)
# <--- Your code here to set up the TFLite converter with INT8 quantization --->
# raise NotImplementedError

import tensorflow as tf
from tqdm import tqdm

# ------------------------------
# FULL INTEGER QUANTIZATION
# ------------------------------

# Define a representative generator required for calibrating quantization
# Note: input data must match model input shape and dtype (float32)
def representative_data_gen(representative_size=256):
    for x_batch, _ in val_dataset.take(representative_size).batch(1):
        yield [x_batch]

# Create the converter from the Keras model
converter = tf_lite.TFLiteConverter.from_keras_model(stripped_model)

# Enable default optimizations (includes quantization)
converter.optimizations = [tf_lite.Optimize.DEFAULT]

# Provide representative dataset for calibration
converter.representative_dataset = representative_data_gen

# Specify that only int8 operations are to be used in the model
converter.target_spec.supported_ops = [tf_lite.OpsSet.TFLITE_BUILTINS_INT8]

# Set input and output tensor types to int8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Convert and save the INT8 quantized model
tflite_model_int8 = converter.convert()
with open("pruned_quantized_student_model.tflite", "wb") as f:
    f.write(tflite_model_int8)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [29]:
converter = tf_lite.TFLiteConverter.from_keras_model(student_model)
student_model = converter.convert()
with open("student_model.tflite", "wb") as f:
    f.write(student_model)


In [30]:
converter = tf_lite.TFLiteConverter.from_keras_model(teacher_model)
teacher_model = converter.convert()
with open("teacher_model.tflite", "wb") as f:
    f.write(teacher_model)

In [42]:
import tensorflow as tf
import numpy as np
import os

# Compare model sizes
model_files = [
    "teacher_model.tflite",
    "student_model.tflite",
    "pruned_quantized_student_model.tflite",
]

print("\nModel Sizes (KB):")
for file in model_files:
    print(f"{file}: {os.path.getsize(file) / 1024:.2f} KB")

# Evaluate accuracy for sparse TFLite model
def evaluate_tflite_model(tflite_model_path, x_test, y_test, batch_size=32):
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)

    # Resize input tensor to accommodate the batch size
    input_details = interpreter.get_input_details()[0]
    interpreter.resize_tensor_input(input_details['index'], [batch_size, 28, 28, 1])

    interpreter.allocate_tensors()

    # Re-fetch details after allocation
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    dtype = input_details['dtype']

    n_batch = len(x_test) // batch_size
    correct_predictions = 0

    for i in range(n_batch):
        # 1. Extract the current batch
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        batch_x = x_test[start_idx:end_idx]
        batch_y = y_test[start_idx:end_idx]

        # 2. Handle Quantization for int8/uint8
        if dtype in [np.int8, np.uint8]:
            scale, zero_point = input_details['quantization']
            batch_x = (batch_x / scale + zero_point).astype(dtype)
        else:
            batch_x = batch_x.astype(dtype)

        # 3. Run Inference
        interpreter.set_tensor(input_details['index'], batch_x)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details['index'])

        # 4. Dequantize if output is integer
        if output_details['dtype'] in [np.int8, np.uint8]:
            scale, zero_point = output_details['quantization']
            output_data = (output_data.astype(np.float32) - zero_point) * scale

        # 5. Tally correct predictions
        predicted_classes = np.argmax(output_data, axis=1)
        # y_test is one-hot encoded, convert it to integer labels
        true_classes = np.argmax(batch_y, axis=1)
        correct_predictions += np.sum(predicted_classes == true_classes)

    # Return exactly as requested
    return correct_predictions / (n_batch * batch_size)

# Print accuracies
print("\nModel Accuracies:")
for model in model_files:
  print(f"{model.split('.')[0]} Accuracy: {evaluate_tflite_model(model, x_test, y_test):.4f}")


Model Sizes (KB):
teacher_model.tflite: 2713.46 KB
student_model.tflite: 682.15 KB
pruned_quantized_student_model.tflite: 175.30 KB

Model Accuracies:


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


teacher_model Accuracy: 0.9836
student_model Accuracy: 0.9762
pruned_quantized_student_model Accuracy: 0.9759



---
⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️

### 📸 Lab 2 Submission Reminder

**Please take a screenshot of the above result and include it as part of your Lab 2 submission.**

---


## Summary

- The student model achieves comparable accuracy to the teacher model while being significantly smaller.
- Full integer quantization reduces the student model size further, with a slight accuracy drop.
- Knowledge Distillation enables smaller models to effectively mimic larger, more complex models.